# LLM Uncertainty Monitoring - End-to-End Demo

This notebook demonstrates the full pipeline: model deployment, data preparation,
conformal calibration, online inference with uncertainty routing, batch evaluation,
and uncertainty explainability.

### Prerequisites

Before running this notebook, deploy the infrastructure from the README (Deployment steps 1-5).

Verify `ml.g5.2xlarge` endpoint quota > 0 (Service Quotas > SageMaker).

## 1. Environment Check

In [ ]:
import sys
import os

# Locate the project root (the folder containing src/llm_uncertainty_monitoring)
# by walking up from the current directory. Idempotent: safe to re-run, and
# works whether the kernel starts in examples/ or the project root.
_root = os.getcwd()
while _root != os.path.dirname(_root):
    if os.path.isdir(os.path.join(_root, "src", "llm_uncertainty_monitoring")):
        break
    _root = os.path.dirname(_root)
os.chdir(_root)
sys.path.insert(0, os.path.join(_root, "src"))  # importable library (src layout)
sys.path.insert(0, _root)  # for `scripts.*` imports

import boto3
import json
import time
import llm_uncertainty_monitoring

sts = boto3.client("sts")
identity = sts.get_caller_identity()
session = boto3.session.Session()
REGION = session.region_name or "us-east-1"
ACCOUNT_ID = identity["Account"]
# The auto-generated bucket names are read from the CloudFormation
# stack outputs later (by deploy_model.py, seeded into DynamoDB).

print(f"Account: {ACCOUNT_ID}")
print(f"Region:  {REGION}")

## 2. Deploy Model

In [ ]:
# Deploy SageMaker endpoint (skip this cell if endpoint already exists)
from scripts.deploy_model import main as deploy_model_main

deploy_model_main([
    "--model-id", "meta-textgeneration-llama-3-8b",
    "--model-version", "3.0.0",
    "--instance-type", "ml.g5.2xlarge",
    "--accept-eula",
])

In [ ]:
# Seed the human review routing config in DynamoDB
# (connects the inference handler to the SQS queue created by CloudFormation)
from llm_uncertainty_monitoring.config.dynamo_store import put_config

sqs_arn = f'arn:aws:sqs:{REGION}:{ACCOUNT_ID}:LLMUncertainty-HumanReview'
put_config(values={
    'human_review_destination_arn': sqs_arn,
    'human_review_destination_type': 'sqs',
})
print(f'Routing config seeded: {sqs_arn}')

In [ ]:
# Read endpoint + auto-generated bucket names from DynamoDB.
# deploy_model.py seeds these from the CloudFormation stack outputs.
from llm_uncertainty_monitoring.config.dynamo_store import load_config, reset_cache

reset_cache()
config = load_config(strict=False)
ENDPOINT_NAME = config["endpoint_name"]
DATA_BUCKET = config["data_bucket_name"]
print(f"Endpoint:    {ENDPOINT_NAME}")
print(f"Data bucket: {DATA_BUCKET}")

## 3. Prepare Data

In [ ]:
SUBJECTS = ["econometrics", "marketing", "management", "sociology"]

from scripts.prepare_data import main as prepare_data_main

prepare_data_main([
    "--endpoint-name", ENDPOINT_NAME,
    "--s3-uri-prefix", f"s3://{DATA_BUCKET}/data",
    "--subjects", *SUBJECTS,
    "--difficult-per-subject", "10",
    "--calibration-fraction", "0.5",
    "--seed", "0",
])

## 4. Calibration — Compute q_hat

In [ ]:
from scripts.run_calibration import main as calibration_main

calibration_main([
    "--endpoint-name", ENDPOINT_NAME,
    "--cal-s3-uri", f"s3://{DATA_BUCKET}/data/cal.jsonl",
    "--output-s3-uri", f"s3://{DATA_BUCKET}/calibration",
])

In [ ]:
from llm_uncertainty_monitoring.config.dynamo_store import load_config, reset_cache

reset_cache()
config = load_config()
Q_HAT = float(config["conformal_threshold"])
TARGET_COVERAGE = float(config["target_coverage"])
print(f"q_hat = {Q_HAT:.4f} (target_coverage = {TARGET_COVERAGE})")

## 5. Online Inference — Certain vs Uncertain

Call the inference handler in-process (same logic as the deployed Lambda).

In [ ]:
from llm_uncertainty_monitoring.inference_handler import _reset_for_testing, lambda_handler
_reset_for_testing()

# Certain prediction (clear-cut question)
certain_event = {
    "question": "What is GDP?",
    "choices": [
        "Total market value of all final goods and services produced in a country",
        "The total amount of money in circulation",
        "The government's annual budget",
        "The stock market index",
    ],
}

# Uncertain prediction (ambiguous econometrics question)
uncertain_event = {
    "question": "Which of the following statements are true concerning a triangular system?",
    "choices": [
        "(ii) and (iv) only",
        "(i) and (iii) only",
        "(i), (ii), and (iii) only",
        "(i), (ii), (iii), and (iv)",
    ],
}

for label, event in [("CERTAIN", certain_event), ("UNCERTAIN", uncertain_event)]:
    result = lambda_handler(event, context=None)
    body = result["body"]
    print(f"\n--- {label} ---")
    print(f"  prediction_set:    {body['prediction_set']}")
    print(f"  set_size:          {body['set_size']}")
    print(f"  routing_decision:  {body['routing_decision']}")
    print(f"  probabilities:     {[f'{p:.3f}' for p in body['probabilities']]}")

## 6. Batch Evaluation — 3 Batches with Drift Simulation

Run 3 batches with 5-minute gaps. Batch 3 includes the difficult questions to
simulate rising uncertainty. The CloudWatch alarm should trigger on batch 3.

In [ ]:
from scripts.run_batch_evaluation import main as batch_eval_main

print('=== Batch 1/3: test rows 0-114 (normal) ===')
batch_eval_main([
    "--input-s3-uri", f"s3://{DATA_BUCKET}/data/test.jsonl",
    "--s3-output-bucket", DATA_BUCKET,
    "--s3-output-prefix", "batch",
    "--max-rows", "115",
])

print('\nWaiting 5 min for CloudWatch period separation...')
time.sleep(300)

print('\n=== Batch 2/3: test rows 115-229 (normal) ===')
batch_eval_main([
    "--input-s3-uri", f"s3://{DATA_BUCKET}/data/test.jsonl",
    "--s3-output-bucket", DATA_BUCKET,
    "--s3-output-prefix", "batch",
    "--skip-rows", "115",
    "--max-rows", "115",
])

print('\nWaiting 5 min for CloudWatch period separation...')
time.sleep(300)

print('\n=== Batch 3/3: remaining test + ALL difficult rows ===')
batch_eval_main([
    "--input-s3-uri", f"s3://{DATA_BUCKET}/data/test.jsonl",
    "--difficult-s3-uri", f"s3://{DATA_BUCKET}/data/difficult.jsonl",
    "--s3-output-bucket", DATA_BUCKET,
    "--s3-output-prefix", "batch",
    "--skip-rows", "230",
])

print('\nBatch evaluation complete. Check CloudWatch for alarm trigger.')

In [ ]:
import pandas as pd

# Load all 3 batch results and display comparison table
s3 = boto3.client("s3")
response = s3.list_objects_v2(Bucket=DATA_BUCKET, Prefix="batch/")
batch_keys = sorted(obj["Key"] for obj in response.get("Contents", []))[-3:]

all_aggregates = []
for key in batch_keys:
    obj = s3.get_object(Bucket=DATA_BUCKET, Key=key)
    batch_data = json.loads(obj["Body"].read())
    
    certain_records = [r for r in batch_data["records"] if r["is_certain"]]
    uncertain_records = [r for r in batch_data["records"] if not r["is_certain"]]
    certain_acc = sum(r["top1_correct"] for r in certain_records) / len(certain_records) if certain_records else 0
    uncertain_acc = sum(r["top1_correct"] for r in uncertain_records) / len(uncertain_records) if uncertain_records else 0
    
    agg = batch_data["aggregates"]
    agg["CertainAccuracy"] = certain_acc
    agg["UncertainAccuracy"] = uncertain_acc
    agg["AccuracyGap"] = certain_acc - uncertain_acc
    all_aggregates.append(agg)

df = pd.DataFrame(all_aggregates, index=["Batch 1 (normal)", "Batch 2 (normal)", "Batch 3 (+ difficult)"]).T
df

## 7. Uncertainty Explanation — Differential Word Cloud

In [ ]:
from llm_uncertainty_monitoring.uncertainty_explainer import compare_certain_uncertain
from IPython.display import Image, display

# Concatenate records from all 3 batches
all_records = []
for key in batch_keys:
    obj = s3.get_object(Bucket=DATA_BUCKET, Key=key)
    data = json.loads(obj["Body"].read())
    all_records.extend(data["records"])
print(f"Total records for XAI: {len(all_records)}")

comparison = compare_certain_uncertain(
    records=all_records,
    top_n=20,
    output_local_path="./wordcloud_comparison.png",
)
display(Image(filename=comparison["output_path"]))

print(f"\nCertain: {comparison['n_certain']} predictions | Uncertain: {comparison['n_uncertain']} predictions")
print("\nWords most associated with UNCERTAIN predictions:")
for word, ratio in comparison["top_uncertain_differential"]:
    print(f"  {word}: {ratio:.0%}")

print("\nWords most associated with CERTAIN predictions:")
for word, ratio in comparison["top_certain_differential"]:
    print(f"  {word}: {ratio:.0%}")

## 8. Cleanup

Delete the SageMaker endpoint to stop incurring GPU costs.
The CloudFormation stack resources remain for inspection.

In [ ]:
from llm_uncertainty_monitoring.model_deployer import ModelDeployer

ModelDeployer().delete_endpoint(ENDPOINT_NAME)
print(f"Deleted endpoint: {ENDPOINT_NAME}")
print("\nTo delete the full CloudFormation stack:")
print("  aws cloudformation delete-stack --stack-name llm-uncertainty-monitoring")